# L10 · Assignment — Ship a RAG shopping assistant

> *Marcus has approved the project. Your task: build a working RAG shopping assistant that can answer customer questions about the NorthStar catalogue. Then evaluate it on a small benchmark and identify two specific improvement areas.*

In this assignment you will:

**Part A.** Implement a `RAGSystem` from scratch (you can adapt NB 04's class). The system should:
- Retrieve top-K relevant products by semantic similarity
- Pass them to an LLM with a constrained system prompt
- Return both the answer AND the retrieved products (audit trail)

**Part B.** Run the system on a 5-query benchmark and grade the output.

**Part C.** Identify two specific weaknesses of your RAG system and describe how you would fix them.

Total runtime: ~5-10 minutes.

---

## Setup

In [1]:
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(1)
torch.manual_seed(0)

df = pd.read_csv('data/northstar_catalogue.csv')
print(f"Catalogue: {len(df)} products")

retriever = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')
llm = AutoModelForCausalLM.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')
print('Models loaded.')

Catalogue: 76 products


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9103.64it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3665.82it/s]

Models loaded.


---

## Part A — Implement your RAGSystem

The class should have at least: `__init__`, `retrieve`, `ask`. Adapt NB 04's class or write your own.

In [2]:
class RAGSystem:
    """Shopping assistant: retrieve relevant products, then generate a grounded answer."""

    def __init__(self, df, retriever, llm, tokenizer):
        self.df = df.reset_index(drop=True).copy()
        self.retriever = retriever
        self.llm = llm
        self.tok = tokenizer
        docs = (self.df['name'] + ' — ' + self.df['description']
                + ' (£' + self.df['price_gbp'].astype(str) + ')').tolist()
        self.embeddings = self.retriever.encode(docs, show_progress_bar=False)

    def retrieve(self, query, top_k=5):
        q_emb = self.retriever.encode([query])
        sims = cosine_similarity(q_emb, self.embeddings)[0]
        idx = np.argsort(-sims)[:top_k]
        return self.df.iloc[idx].copy().assign(similarity=sims[idx]).reset_index(drop=True)

    def ask(self, query, top_k=5, max_new_tokens=180):
        retrieved = self.retrieve(query, top_k=top_k)
        # Format retrieved products into the system prompt
        lines = [f"- [{r['product_id']}] {r['name']} ({r['category']}, £{r['price_gbp']}): {r['description']}"
                 for _, r in retrieved.iterrows()]
        catalogue_block = '\n'.join(lines)
        sys_msg = (
            "You are a helpful retail shopping assistant for NorthStar. "
            "ONLY recommend products from the catalogue below. "
            "If nothing matches, say so honestly. Be concise.\n\n"
            f"CATALOGUE:\n{catalogue_block}"
        )
        messages = [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': query},
        ]
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = self.tok(prompt, return_tensors='pt')['input_ids']
        out = self.llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=self.tok.eos_token_id)
        answer = self.tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        return {'query': query, 'answer': answer, 'retrieved': retrieved}

rag = RAGSystem(df, retriever, llm, tokenizer)
print('RAGSystem ready.')

RAGSystem ready.


---

## Part B — Benchmark the system

Run 5 plausible customer queries through the system. Evaluate each on three criteria.

In [3]:
BENCHMARK = [
    "I need a warm winter coat for under £200",
    "Something to wear to a summer beach holiday",
    "What's a smart but comfortable shirt for the office?",
    "Recommend a gym outfit for cold-weather running",
    "I'm looking for a wedding-guest dress",
]

results = []
for q in BENCHMARK:
    t0 = time.time()
    r = rag.ask(q)
    elapsed = time.time() - t0
    results.append({'query': q, 'answer': r['answer'], 'retrieved': r['retrieved'], 'elapsed': elapsed})

# Pretty-print each result
for i, r in enumerate(results, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}: {r['query']}")
    print(f"\n[Retrieved top-3]")
    for _, row in r['retrieved'].head(3).iterrows():
        print(f"  - {row['name']:<35s} ({row['category']:<10s} £{row['price_gbp']:>3d})")
    print(f"\nA: {r['answer']}")
    print(f"[{r['elapsed']:.1f}s]")


Q1: I need a warm winter coat for under £200

[Retrieved top-3]
  - Highland Trench Coat                (coat       £195)
  - Frost Linen Shirt                   (shirt      £ 55)
  - Ember Quilted Jacket                (coat       £110)

A: I'm sorry, but as a retail shopping assistant, I don't have the ability to browse the internet or provide product recommendations. I'm here to assist with your shopping needs.
[7.6s]

Q2: Something to wear to a summer beach holiday

[Retrieved top-3]
  - Cassia Maxi Gown                    (dress      £ 65)
  - Frost Linen Shirt                   (shirt      £ 55)
  - Driftwood Straw Hat                 (accessory  £ 38)

A: - [P0003] Cassia Maxi Gown (dress, £65): Flowing full-length gown with adjustable straps. Beach holiday essential.
- [P0014] Frost Linen Shirt (shirt, £55): Breathable linen in pale ice blue. Vacation packing essential.
- [P0055] Driftwood Straw Hat (accessory, £38): Wide-brim straw sun hat with ribbon trim. Beach essential.
-

### B.1 · Grade each answer

For each of the 5 answers, rate on three criteria (write `1` = good, `0` = problem):

- **Grounded** — does the answer reference products that are in the retrieved set?
- **Relevant** — does it match what the customer asked for?
- **Honest** — if no good product exists, does it say so (vs. forcing a recommendation)?

In [ ]:
# Fill in your grades manually based on the printed answers above.
# Replace each None with 1 (good), 0 (problem), or 0.5 (partial credit).
# The cell will refuse to score until you've graded every row.

grades = [
    # query, grounded, relevant, honest
    (BENCHMARK[0], None, None, None),
    (BENCHMARK[1], None, None, None),
    (BENCHMARK[2], None, None, None),
    (BENCHMARK[3], None, None, None),
    (BENCHMARK[4], None, None, None),
]

# Validate that you've filled them in
if any(v is None for _, *vs in grades for v in vs):
    print("⚠️  Some grades are still None. Fill them in based on the answers printed above.")
    print("    1 = good, 0 = problem, 0.5 = partial credit.")
    print("    Then re-run this cell.")
else:
    print(f"{'Query':45s} {'Ground':>7s} {'Relev':>7s} {'Honest':>7s}")
    print('-' * 75)
    totals = [0, 0, 0]
    for q, g, r, h in grades:
        print(f"  {q[:43]:45s} {g:>7} {r:>7} {h:>7}")
        totals[0] += g; totals[1] += r; totals[2] += h

    n = len(grades)
    print(f"\n{'TOTAL (out of '+str(n)+')':45s} {totals[0]:>7} {totals[1]:>7} {totals[2]:>7}")
    overall = sum(totals) / (3 * n)
    print(f"\nOverall quality score: {overall:.2f} / 1.00")
    print(f"\nTarget ≥ 0.7 for a shippable demo? {'✅ PASS' if overall >= 0.7 else '❌ Investigate why'}")

---

## Part C — Identify two improvement areas

Looking at your grading, find two specific weaknesses of your RAG system. For each:

1. **Describe** the weakness with a specific example from your benchmark
2. **Propose** a concrete fix (one to two sentences — don't just say "use a bigger model"; what specifically?)

### Weakness 1

**Description:** *(your answer)*

**Fix:** *(your answer)*

### Weakness 2

**Description:** *(your answer)*

**Fix:** *(your answer)*

### Optional reflection

Of the techniques in NB 04 Extensions (hallucination check, LLM re-rank), which would help your benchmark queries most? Why?

*(your answer)*

---

## Submission checklist

- [ ] Part A — `RAGSystem` implemented with retrieve + ask methods
- [ ] Part B — 5 benchmark queries run, answers printed
- [ ] Part B — Manual grading filled in (3 criteria × 5 queries)
- [ ] Part B target — Overall quality ≥ 0.7
- [ ] Part C — Two weaknesses identified with concrete fixes
- [ ] Part C — Optional reflection answered

Save the notebook with outputs and submit.